# Milestone 2: RAG Pipeline Exploration

This notebook documents our exploration process for building the RAG pipeline on the All_Beauty Amazon Reviews 2023 dataset.

It covers:
1. Testing BM25, Semantic, and Hybrid retrievers
2. Inspecting the context block passed to the LLM
3. Comparing prompt variants V1 and V2
4. Running the full RAG pipeline end to end

**Note:** Full implementation lives in `src/rag_pipeline.py`. This notebook shows the key exploration steps that informed our design decisions.

## Setup

In [1]:
import sys
import pickle
import torch
from pathlib import Path

# Add repo root to path so we can import from src/
REPO_ROOT = Path().resolve().parent
sys.path.append(str(REPO_ROOT))

from src.bm25 import load_bm25_index, search_bm25
from src.semantic import load_semantic_index, search_semantic
from sentence_transformers import SentenceTransformer

CORPUS_METADATA_PATH = REPO_ROOT / "data" / "processed" / "corpus_metadata.pkl"

# Check device
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [2]:
# Load indexes and metadata
print("Loading BM25 index...")
bm25, metadata, _ = load_bm25_index()

print("Loading FAISS index...")
faiss_index = load_semantic_index()

print("Loading sentence transformer model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"\nCorpus size: {len(metadata)} documents")

Loading BM25 index...
Loading BM25 index from /Users/jangaya/Desktop/MDS/Classes/block_6/575/DSCI_575_project_yasieft_purityj/data/processed/bm25_index.pkl...
Loading corpus metadata from /Users/jangaya/Desktop/MDS/Classes/block_6/575/DSCI_575_project_yasieft_purityj/data/processed/corpus_metadata.pkl...
Loaded index with 112590 documents
Loading FAISS index...
Loading FAISS index from /Users/jangaya/Desktop/MDS/Classes/block_6/575/DSCI_575_project_yasieft_purityj/data/processed/faiss_index.faiss...
Loaded FAISS index with 112590 vectors
Loading sentence transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Corpus size: 112590 documents


---
## Section 1: Comparing Retrievers

We test BM25, Semantic, and Hybrid retrieval on the same query to understand their differences.
This directly informed our decision to use Hybrid retrieval in the RAG pipeline.

In [3]:
# Helper to print results cleanly
def print_results(results, method_name):
    print(f"\n{'='*60}")
    print(f"{method_name} Top 3 Results")
    print(f"{'='*60}")
    for r in results:
        print(f"[{r['rank']}] {r['title'][:70]}")
        print(f"     Score: {r['score']}")
        print()

In [4]:
# Test query 1: keyword-based — BM25 should do well
query = "vitamin C serum"
print(f"Query: '{query}'\n")

bm25_results = search_bm25(query, bm25, metadata, top_k=3)
sem_results = search_semantic(query, faiss_index, metadata, model, top_k=3)

print_results(bm25_results, "BM25")
print_results(sem_results, "Semantic")

Query: 'vitamin C serum'

Query tokens: ['vitamin', 'serum']

BM25 Top 3 Results
[1] Springs Organic Vitamin C Serum For Your Face - Vitamin C Serum|Organi
     Score: 17.2543

[2] Best Vitamin C Serum With Retinol, CoQ10, Matrixyl 3000 Hyaluronic Aci
     Score: 15.9507

[3] Serum Sensation Vitamin C Serum with Hyaluronic Acid, Organic Anti-Agi
     Score: 15.4532


Semantic Top 3 Results
[1] PURE VITAMIN C SERUM
     Score: 0.7282

[2] Liz K Super First C Serum Pure Vitamin C 13%
     Score: 0.7243

[3] Vitamin C Serum 2 fl. oz with L'Ascorbic Acid - Facial Skin Care & Ant
     Score: 0.7078



In [5]:
# Test query 2: intent-based — Semantic should do better
query = "something to keep my face hydrated all day"
print(f"Query: '{query}'\n")

bm25_results = search_bm25(query, bm25, metadata, top_k=3)
sem_results = search_semantic(query, faiss_index, metadata, model, top_k=3)

print_results(bm25_results, "BM25")
print_results(sem_results, "Semantic")

print("\nObservation: BM25 matches on 'my' and 'face' returning irrelevant results.")
print("Semantic search correctly understands the hydration intent.")

Query: 'something to keep my face hydrated all day'

Query tokens: ['something', 'keep', 'my', 'face', 'hydrated', 'all', 'day']

BM25 Top 3 Results
[1] Grace My Face Minerals Powder Me Louder Soothing Redness Control Found
     Score: 17.6282

[2] Not My Mama's Bubbling Girls Face Wash All Natural Daily Cleanser 6 oz
     Score: 15.23

[3] Kiss My Face Organic Lip Balm Pineapple Coconut - .18 oz. - 3 Pack!
     Score: 15.1188


Semantic Top 3 Results
[1] One Drop of Our All Natural Hyaluronic Acid Serum Will Soothe and Hydr
     Score: 0.6603

[2] Face Oasis Cleansing Water
     Score: 0.6225

[3] Hydrating Face Mask For Men and Women By ERH - Top Rated Formula Full 
     Score: 0.6086


Observation: BM25 matches on 'my' and 'face' returning irrelevant results.
Semantic search correctly understands the hydration intent.


In [6]:
# Hybrid retriever: merge and deduplicate BM25 + Semantic results
def hybrid_retrieve(query, bm25, faiss_index, metadata, model, top_k=5):
    sem_results = search_semantic(query, faiss_index, metadata, model, top_k)
    bm25_results = search_bm25(query, bm25, metadata, top_k)

    seen = set()
    combined = []
    for doc in sem_results + bm25_results:
        title = doc.get("title", "")
        if title not in seen:
            seen.add(title)
            combined.append(doc)

    return combined[:top_k]

query = "something to keep my face hydrated all day"
hybrid_results = hybrid_retrieve(query, bm25, faiss_index, metadata, model, top_k=3)
print_results(hybrid_results, "Hybrid")

print("\nObservation: Hybrid combines the strengths of both methods.")
print("For keyword queries it inherits BM25 precision.")
print("For intent queries it inherits Semantic understanding.")

Query tokens: ['something', 'keep', 'my', 'face', 'hydrated', 'all', 'day']

Hybrid Top 3 Results
[1] One Drop of Our All Natural Hyaluronic Acid Serum Will Soothe and Hydr
     Score: 0.6603

[2] Face Oasis Cleansing Water
     Score: 0.6225

[3] Hydrating Face Mask For Men and Women By ERH - Top Rated Formula Full 
     Score: 0.6086


Observation: Hybrid combines the strengths of both methods.
For keyword queries it inherits BM25 precision.
For intent queries it inherits Semantic understanding.


---
## Section 2: Inspecting the Context Block

Before passing documents to the LLM, we build a structured context block.
Here we inspect what the LLM actually receives — this helped us identify a bug
where retrieval scores were mislabeled as ratings.

In [7]:
import duckdb

PARQUET_PATH = REPO_ROOT / "data" / "processed" / "All_Beauty.parquet"
conn = duckdb.connect()

def get_review(parent_asin):
    query = f"""
        SELECT text FROM read_parquet('{PARQUET_PATH}')
        WHERE asin = ? LIMIT 1
    """
    result = conn.execute(query, [parent_asin]).fetchone()
    return result[0] if result else "No review found"

def build_context(docs):
    context_blocks = []
    for doc in docs:
        block = f"""
        Product ASIN: {doc.get('parent_asin', 'N/A')}
        Title: {doc.get('title', 'N/A')}
        Rating: {doc.get('score', 'N/A')}
        Review: {doc.get('review_text', get_review(doc['parent_asin'])[:200])}
        """
        context_blocks.append(block.strip())
    return "\n\n".join(context_blocks)

In [8]:
# Inspect context for a sample query
query = "gentle cleanser for sensitive skin"
docs = hybrid_retrieve(query, bm25, faiss_index, metadata, model, top_k=3)
context = build_context(docs)

print("Context block passed to LLM:")
print("-" * 60)
print(context)
print("-" * 60)
print()
print("BUG IDENTIFIED: 'Rating' field shows cosine similarity score (e.g. 0.67)")
print("not the actual product star rating (1-5).")
print("This caused the LLM to misinterpret scores as star ratings in early tests.")
print()
print("FIX: Use doc.get('average_rating') instead of doc.get('score') for the Rating field.")

Query tokens: ['gentle', 'cleanser', 'sensitive', 'skin']
Context block passed to LLM:
------------------------------------------------------------
Product ASIN: B019JZNIC8
        Title: Jason Gentle Basics Facial Cleanser, 16 Fluid Ounce (Pack of 6)
        Rating: 0.6904
        Review: I’ve used this product for years. But can’t seem to find it local anymore. So this case will last a long time but worth it in the end.

Product ASIN: B01JA7TMSO
        Title: Calming Chamomile Daily Face Cleanser for Sensitive Skin with Shea Butter & Aloe Vera - Gently Remove Make-Up, Dirt & Oil Without Irritation.
        Rating: 0.6806
        Review: I really like it already got my next one

Product ASIN: B001P4ZTFG
        Title: Serious Skin Care Glycolic Cleanser
        Rating: 0.6676
        Review: I had formerly used this product in the late-90's when it was first introduced to the market. I liked it very well then, and found it highly effective in tone and clarity until sudden onset of ba

---
## Section 3: Prompt Variant Comparison (V1 vs V2)

We tested two system prompt variants on the same query.
V1 uses strict rules, V2 uses softer expert-style instructions.
The outputs below informed our decision to use V1 as the default.

In [9]:
SYSTEM_PROMPT_V1 = """
You are a helpful Amazon shopping assistant.

STRICT RULES:
- Answer ONLY using the provided reviews
- Do NOT make up information
- Do NOT repeat the question or instructions
- Do NOT output words like "assistant" or tags like "<think>"
- Do NOT generate numbered lists (1, 2, 3, etc.)
- Keep the answer short and clear (2-4 sentences max)
- Always cite product ASINs when relevant

Output ONLY the final answer. No extra text.
"""

SYSTEM_PROMPT_V2 = """
You are an expert product recommendation assistant.

TASK:
Use the provided reviews to answer the question.

RULES:
- Base your answer ONLY on the reviews
- Do NOT invent information
- Do NOT repeat the prompt or instructions
- Avoid numbered or bullet lists unless necessary
- Keep the answer concise and readable
- Mention product ASINs when relevant

Output ONLY the final answer. No explanations.
"""

def build_prompt(query, context, version="v1"):
    system_prompt = SYSTEM_PROMPT_V1 if version == "v1" else SYSTEM_PROMPT_V2
    return f"""
    {system_prompt}

    Customer Reviews Are:
    {context}

    Question is: {query}

    FINAL ANSWER:
    """

print("Prompt variants defined.")

Prompt variants defined.


In [10]:
# Show the actual outputs we observed when testing V1 vs V2
# (outputs captured from terminal runs - see results/milestone2_discussion.md)

query = "Product for dark spots and uneven skin tone"

v1_output = """
* B00B0ZIQ64 (BREMENN CLINICAL Dark Spot Eraser - Compound for Visibly Reducing Dark Spots)
* B099K3QQ8M (Intimate Area Dark Spot Corrector - Skincare Cream for Discoloration)

B00B0ZIQ64 is the most relevant product for reducing dark spots and correcting uneven skin tone.
"""

v2_output = """
1. B00B0ZIQ64
2. B01EVFIHW2
3. B099K3QQ8M
Final answer: B00B0ZIQ64
Final answer: B01EVFIHW2
Final answer: B099K3QQ8M
Final answer: B00B0ZIQ64  ...(repeated)
"""

print(f"Query: '{query}'")
print()
print("V1 Output:")
print("-" * 50)
print(v1_output)
print()
print("V2 Output:")
print("-" * 50)
print(v2_output)
print()
print("Finding: V1 produced an acceptable answer with reasoning.")
print("V2 repeated ASINs with no product names or reasoning.")
print("V1 selected as default for the RAG pipeline.")

Query: 'Product for dark spots and uneven skin tone'

V1 Output:
--------------------------------------------------

* B00B0ZIQ64 (BREMENN CLINICAL Dark Spot Eraser - Compound for Visibly Reducing Dark Spots)
* B099K3QQ8M (Intimate Area Dark Spot Corrector - Skincare Cream for Discoloration)

B00B0ZIQ64 is the most relevant product for reducing dark spots and correcting uneven skin tone.


V2 Output:
--------------------------------------------------

1. B00B0ZIQ64
2. B01EVFIHW2
3. B099K3QQ8M
Final answer: B00B0ZIQ64
Final answer: B01EVFIHW2
Final answer: B099K3QQ8M
Final answer: B00B0ZIQ64  ...(repeated)


Finding: V1 produced an acceptable answer with reasoning.
V2 repeated ASINs with no product names or reasoning.
V1 selected as default for the RAG pipeline.


In [11]:
# Summary of V1 vs V2 across all 5 queries
import pandas as pd

comparison = pd.DataFrame({
    "Query": [
        "Best moisturizers for dry skin",
        "Something to keep face hydrated",
        "Product for dark spots",
        "Anti aging cream under $30",
        "Gift for skincare lover"
    ],
    "V1 Result": [
        "Listed products + ASINs (cut off at token limit)",
        "Echoed raw context block",
        "Correct recommendation + brief reasoning (BEST)",
        "Listed ASINs only, no product names",
        "Repeated same sentence 3x + </think> tag leak"
    ],
    "V2 Result": [
        "Leaked full <think> reasoning block",
        "Repeated filler phrase 12 times",
        "Listed ASINs only, no reasoning (WORSE than V1)",
        "Listed ASINs repeatedly (same failure)",
        "Echoed raw review snippets"
    ],
    "Winner": ["V1", "V1", "V1", "Tie", "V1"]
})

print(comparison.to_string(index=False))
print()
print("Conclusion: V1 outperformed V2 on 4/5 queries.")
print("Both prompts failed on most queries due to model size (0.8B params).")
print("Prompt engineering cannot compensate for insufficient model capacity.")

                          Query                                        V1 Result                                       V2 Result Winner
 Best moisturizers for dry skin Listed products + ASINs (cut off at token limit)             Leaked full <think> reasoning block     V1
Something to keep face hydrated                         Echoed raw context block                 Repeated filler phrase 12 times     V1
         Product for dark spots  Correct recommendation + brief reasoning (BEST) Listed ASINs only, no reasoning (WORSE than V1)     V1
     Anti aging cream under $30              Listed ASINs only, no product names          Listed ASINs repeatedly (same failure)    Tie
        Gift for skincare lover    Repeated same sentence 3x + </think> tag leak                      Echoed raw review snippets     V1

Conclusion: V1 outperformed V2 on 4/5 queries.
Both prompts failed on most queries due to model size (0.8B params).
Prompt engineering cannot compensate for insufficient model capacit

---
## Section 4: Full RAG Pipeline End to End

We run the complete pipeline: query → hybrid retrieval → context → prompt → LLM answer.
This uses `rag_pipeline.py` directly and shows real outputs from our terminal runs.

In [12]:
from src.rag_pipeline import rag_pipeline

# Run one query end to end
# Note: LLM inference takes ~20-30 seconds on MPS, longer on CPU
query = "What are the best moisturizers for dry skin?"
print(f"Running RAG pipeline for: '{query}'")
print("(This may take 20-30 seconds on MPS...)\n")

answer, prompt, docs = rag_pipeline(query, mode="Hybrid", prompt_version="v1")

print("Retrieved Documents:")
print("-" * 50)
for i, doc in enumerate(docs, 1):
    print(f"[{i}] {doc['title'][:70]}")
    print(f"     ASIN: {doc.get('parent_asin', 'N/A')}")

print()
print("Generated Answer:")
print("-" * 50)
print(answer)

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading FAISS index from /Users/jangaya/Desktop/MDS/Classes/block_6/575/DSCI_575_project_yasieft_purityj/data/processed/faiss_index.faiss...
Loaded FAISS index with 112590 vectors


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading BM25 index from /Users/jangaya/Desktop/MDS/Classes/block_6/575/DSCI_575_project_yasieft_purityj/data/processed/bm25_index.pkl...
Loading corpus metadata from /Users/jangaya/Desktop/MDS/Classes/block_6/575/DSCI_575_project_yasieft_purityj/data/processed/corpus_metadata.pkl...
Loaded index with 112590 documents
Running RAG pipeline for: 'What are the best moisturizers for dry skin?'
(This may take 20-30 seconds on MPS...)

Query tokens: ['what', 'best', 'moisturizers', 'dry', 'skin']


Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved Documents:
--------------------------------------------------
[1] DayTime Moisturizer for Dry Skin
     ASIN: B01M1A0H4Z
[2] Psoriasis & Eczema Cream 2oz Organic Advanced Healing Moisturizer with
     ASIN: B01A9DVW8Q
[3] Neolia Hydra-Screen Apricot Oil Soap for Dry Skin, 4.5 oz
     ASIN: B005WT11D4
[4] Physicians Formula Elastin & Collagen Moisture Lotion, 4 oz
     ASIN: B00LTJV320
[5] BEST Night Moisturizer for Face & Neck -Deeply Hydrates, Fights Wrinkl
     ASIN: B014RRQEC4

Generated Answer:
--------------------------------------------------
Answer:
</think>

The best moisturizers for dry skin are DayTime Moisturizer for Dry Skin (ASIN: B01M1A0H4Z) and Psoriasis & Eczema Cream 2oz Organic Advanced Healing Moisturizer with Shea Butter, Coconut Oil, Aloe Vera & Manuka Honey (ASIN: B01A9DVW8Q). Other products like Neolia Hydra-Screen Apriot oil Soap and Physicians Formula Elastin & Collagen Moisture Lotion are also recommended based on customer ratings.


In [13]:
# Show actual outputs observed across all 5 evaluation queries
# (captured from terminal - full analysis in results/milestone2_discussion.md)

observed_outputs = [
    {
        "query": "What are the best moisturizers for dry skin?",
        "answer": "Listed DayTime Moisturizer, Psoriasis & Eczema Cream, Neolia Hydra-Screen Soap, Physicians Formula Lotion, BEST Night Moisturizer (cut off at token limit)",
        "accuracy": "Yes",
        "completeness": "No",
        "fluency": "No",
        "note": "Truncated at 200 tokens; numbered list despite V1 prohibition"
    },
    {
        "query": "Something to keep my face hydrated all day",
        "answer": "Echoed raw context block (Hyaluronic Acid Serum, Face Oasis Cleansing Water...)",
        "accuracy": "Partial",
        "completeness": "No",
        "fluency": "No",
        "note": "Retrieval worked; generation failed — model too small"
    },
    {
        "query": "Product for dark spots and uneven skin tone",
        "answer": "BREMENN CLINICAL Dark Spot Eraser (B00B0ZIQ64) identified as most relevant",
        "accuracy": "Yes",
        "completeness": "Partial",
        "fluency": "Partial",
        "note": "Best performing query — simple extraction task"
    },
    {
        "query": "Best anti aging cream under 30 dollars",
        "answer": "Only raw ASINs listed (B010L93H2C, B00FJPMNEM...) — no product names",
        "accuracy": "No",
        "completeness": "No",
        "fluency": "No",
        "note": "Price data removed in preprocessing (84.3% missing); model failed entirely"
    },
    {
        "query": "Gift for someone who loves skincare",
        "answer": "Repeated 'The Body Shop Shea Gift Set is great' 3 times + </think> tag leak",
        "accuracy": "Partial",
        "completeness": "No",
        "fluency": "No",
        "note": "Retrieval worked; generation failed with repetition and tag leak"
    }
]

results_df = pd.DataFrame(observed_outputs)
print(results_df[["query", "accuracy", "completeness", "fluency"]].to_string(index=False))

                                       query accuracy completeness fluency
What are the best moisturizers for dry skin?      Yes           No      No
  Something to keep my face hydrated all day  Partial           No      No
 Product for dark spots and uneven skin tone      Yes      Partial Partial
      Best anti aging cream under 30 dollars       No           No      No
         Gift for someone who loves skincare  Partial           No      No


---
## Summary

**Key findings from this exploration:**

1. **Hybrid retrieval outperforms both BM25 and Semantic alone** — it handles both keyword queries (brand names, product types) and intent-based queries (descriptive, conceptual) in a single system.

2. **The context block has a bug** — retrieval scores (cosine similarity) are mislabeled as ratings, causing the LLM to misinterpret them as star ratings. Fix: use `average_rating` from metadata instead of `score`.

3. **V1 prompt outperforms V2** — stricter instructions produce marginally better outputs at 0.8B scale. However, neither prompt can fully compensate for the model being too small.

4. **Retrieval works well; generation is the bottleneck** — the LLM consistently fails to synthesize retrieved documents into readable answers. Upgrading to Llama3-8B via Groq is the recommended next step for Milestone 3.

5. **Price-constrained queries are unsatisfiable** — price data was removed during Milestone 1 preprocessing (84.3% missing), so no price information exists in the pipeline. A web search tool would be needed to handle this query type.

Full qualitative evaluation with per-query analysis is in `results/milestone2_discussion.md`.